# Week 3 · Part B — Lie Factor audit

This notebook computes Tufte's Lie Factor for graphics A–E using the supplied measurements and `vizlib.lie_factor()`. It then explains the encoding behind each result and tests whether Chart F's dual axes support a valid Lie Factor.

For a change from value $v_0$ to $v_1$, the data effect is $(v_1-v_0)/v_0$. The graphic effect is the proportional change in the encoded ink; for area encodings, the measured linear scale must be raised to the `ink_dimension` power. A Lie Factor near 1 is faithful; values above 1 exaggerate and values below 1 understate.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

WEEK3_DIR = next(
    path for path in (Path.cwd(), Path.cwd() / "week3", Path.cwd().parent / "week3")
    if (path / "assignments" / "vizlib.py").is_file()
)
ASSIGNMENTS_DIR = WEEK3_DIR / "assignments"
DATA_DIR = ASSIGNMENTS_DIR / "data"
sys.path.insert(0, str(ASSIGNMENTS_DIR))

from vizlib import lie_factor, lie_verdict

lie_data = pd.read_csv(DATA_DIR / "case07_lie_factor.csv")
dual_axis_data = pd.read_csv(DATA_DIR / "case07_dual_axis_series.csv")
print(f"Loaded {len(lie_data)} measured-ink cases and {len(dual_axis_data)} monthly revenue/cost rows.")

Loaded 5 measured-ink cases and 6 monthly revenue/cost rows.


## B1 · Compute the measured-ink audit

For each chart, use the supplied `ink_dimension` so area encodings are not mistaken for lengths. The table reports the data effect, graphic effect, Lie Factor, and `vizlib` verdict.

In [2]:
audit_rows = []
for case in lie_data.to_dict("records"):
    lf, data_effect, graphic_effect = lie_factor(
        case["value_start"],
        case["value_end"],
        case["ink_start"],
        case["ink_end"],
        ink_dimension=int(case["ink_dimension"]),
    )
    audit_rows.append({
        "chart_id": case["chart_id"],
        "subject": case["subject"],
        "data_effect": data_effect,
        "graphic_effect": graphic_effect,
        "lie_factor": lf,
        "verdict": lie_verdict(lf),
        "ink_dimension": int(case["ink_dimension"]),
    })

lie_audit = pd.DataFrame(audit_rows).set_index("chart_id")
print("Lie Factor = graphic effect / data effect; honest interval = 0.95 to 1.05")
display(lie_audit.round({"data_effect": 3, "graphic_effect": 3, "lie_factor": 3}))
assert (lie_audit["verdict"] == "honest").sum() == 1

Lie Factor = graphic effect / data effect; honest interval = 0.95 to 1.05


,subject,data_effect,graphic_effect,lie_factor,verdict,ink_dimension
chart_id,,,,,,
A,Corporate tax rate,0.067,5.000,75.00,exaggerates 75.00x,1
B,Federal health budget (Rs bn),0.500,1.250,2.50,exaggerates 2.50x,2
C,Export earnings ($ bn),0.200,0.200,1.00,honest,1
D,Air-pollution index,0.500,0.125,0.25,understates (0.25),1
E,City population (millions),3.000,15.000,5.00,exaggerates 5.00x,2


**B1 result.** The computed Lie Factors are A = 75.00, B = 2.50, C = 1.00, D = 0.25, and E = 5.00. The table uses the `ink_dimension` recorded for each graphic and the toolkit's 0.95–1.05 honesty interval.

## B2 · Which chart is honest, and why are the others wrong?

Chart **C** is the only honest graphic: bars start at zero, and their lengths rise from 5 to 6 cm, matching the data's 20% increase from 25 to 30.

| Chart | What the ink encodes | Why it distorts |
|---|---|---|
| A | Visible bar length from a 29.6% baseline | A 2-point tax change becomes a 4-to-24 mm visible segment change, exaggerating the increase. |
| B | Coin area, scaled 1.5× in both height and width | The visible area grows 2.25× for a 1.5× budget change, exaggerating the change. |
| D | Zero-based bar length | Ink grows only 8-to-9 cm (12.5%) while the index rises 100-to-150 (50%), understating the change. |
| E | Bubble area determined by radius | A 4× radius creates 16× the area for a 4× population, exaggerating the change. |

## B3 · Why B and E require `ink_dimension=2`

For an icon or bubble, the measured linear change is not the displayed area change. Compare each correct area-based Lie Factor with the result produced by incorrectly treating its scale as a length.

In [3]:
dimension_comparison = []
for chart_id in ["B", "E"]:
    case = lie_data.loc[lie_data["chart_id"] == chart_id].iloc[0]
    correct_lf, _, correct_graphic_effect = lie_factor(
        case["value_start"], case["value_end"], case["ink_start"], case["ink_end"],
        ink_dimension=int(case["ink_dimension"]),
    )
    length_only_lf, _, length_only_effect = lie_factor(
        case["value_start"], case["value_end"], case["ink_start"], case["ink_end"],
        ink_dimension=1,
    )
    dimension_comparison.append({
        "chart_id": chart_id,
        "data_dimension": int(case["ink_dimension"]),
        "correct_graphic_effect": correct_graphic_effect,
        "correct_lie_factor": correct_lf,
        "if_misread_as_length_effect": length_only_effect,
        "if_misread_as_length_lie_factor": length_only_lf,
    })
display(pd.DataFrame(dimension_comparison).set_index("chart_id").round(3))
assert np.allclose([row["if_misread_as_length_lie_factor"] for row in dimension_comparison], 1.0)

,data_dimension,correct_graphic_effect,correct_lie_factor,if_misread_as_length_effect,if_misread_as_length_lie_factor
chart_id,,,,,
B,2,1.25,2.5,0.5,1.0
E,2,15.00,5.0,3.0,1.0


**Why dimension 2 matters.** Chart B scales a coin by 1.5 in both height and width, so its area ratio is $1.5^2 = 2.25$. Chart E scales bubble radius by 4, so its area ratio is $4^2 = 16$. If either is incorrectly treated as a length, the computed Lie Factor becomes 1.0 and the distortion disappears on paper. Area grows with the square of its linear scale, so `ink_dimension=2` is essential.

## B4 · Understatement still distorts

Chart D has LF = 0.25: the AQI increases 50%, but the bar length increases only 12.5%. That understates the deterioration and is still a distortion, even though the chart makes the change look smaller rather than larger. A polluter or public authority seeking to minimize concern about worsening air quality would benefit; the measurements alone do not establish who created the graphic or why.

## B5 · Chart F and the dual-axis limitation

A Lie Factor requires one data effect and one comparable graphic effect encoded on a common scale. Calculate the actual monthly margins and check what the independent axes do to the visual ordering; do not force a single Lie Factor onto two unrelated scales.

In [4]:
chart_f = dual_axis_data.copy()
chart_f["margin_pkr_m"] = chart_f["revenue_pkr_m"] - chart_f["cost_pkr_m"]
chart_f["revenue_axis_position"] = (chart_f["revenue_pkr_m"] - 50) / (80 - 50)
chart_f["cost_axis_position"] = (chart_f["cost_pkr_m"] - 40) / (56 - 40)
chart_f["cost_appears_higher"] = chart_f["cost_axis_position"] > chart_f["revenue_axis_position"]
print("Chart F data and visual positions under the prescribed independent axis limits:")
display(chart_f.set_index("month").round(3))
print(f"Months where the dual-axis drawing places cost higher: {chart_f['cost_appears_higher'].sum()} of {len(chart_f)}")
print(f"Months with actual cost above revenue: {(chart_f['cost_pkr_m'] > chart_f['revenue_pkr_m']).sum()} of {len(chart_f)}")
print("Chart F Lie Factor: undefined, not a numeric value (no single shared scale or ink effect).")
assert (chart_f["margin_pkr_m"] > 0).all()
assert chart_f["cost_appears_higher"].iloc[:4].all()

Chart F data and visual positions under the prescribed independent axis limits:


,revenue_pkr_m,cost_pkr_m,margin_pkr_m,revenue_axis_position,cost_axis_position,cost_appears_higher
month,,,,,,
Jan,52,44,8,0.067,0.250,True
Feb,58,46,12,0.267,0.375,True
Mar,61,47,14,0.367,0.438,True
Apr,66,49,17,0.533,0.562,True
May,71,51,20,0.700,0.688,False
Jun,78,53,25,0.933,0.812,False


Months where the dual-axis drawing places cost higher: 4 of 6
Months with actual cost above revenue: 0 of 6
Chart F Lie Factor: undefined, not a numeric value (no single shared scale or ink effect).


**B5 result.** A Lie Factor cannot be computed for Chart F: its two series use separate, independently chosen y-axis ranges, so there is no single ink ratio corresponding to one data effect. The visual ordering can be changed by changing either axis limit. The underlying data show revenue above cost in every month, with positive margins increasing from PKR 8 million in January to PKR 25 million in June; it did not run at a loss in any listed month.